In [1]:
import sympy as sp
import numpy as np
import itertools as it
import flint
import time

In [2]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

project_root = Path("/Users/kristof/main/research/integrable_systems")
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

from NCq3solve.operators import SPIN1, PAULI, LABELS, LABELS_DIP_REAL, LABELS_FULL, LABELS_REAL, PROJECTORS
from NCq3solve.equations import EquationSystem
from NCq3solve.ansatz import Ansatz, hdict_builder
from NCq3solve.families import family, symmetries_for
from NCq3solve.pipeline import Solver

In [3]:
# back-conversion from flint to sympy based on nametable
def flint_to_sympy(eqflint, nametable):
    eq, ctx = eqflint, eqflint.context()
    eqdict = eq.to_dict()
    gens = ctx.gens()
    n = ctx.nvars()
    out = 0
    symbols = {sf: sp.Symbol(f'{nametable[sf]}', real=True) for sf in nametable.keys()}
    for k, c in eqdict.items():
        term = c
        for i in range(n):
            term *= symbols[str(gens[i])]**k[i]
        out += term
    return out

ctx = flint.fmpq_mpoly_ctx.get(('x', 'y', 'z'), ordering='lex')
eq = ctx.from_dict({(0, 0, 3): 2, (0, 1, 2): 1, (1, 1, 1): flint.fmpq(4, 5)})
print(eq.to_dict())
print(ctx.nvars())
nametable = {'x': 'A_0', 'y': 'A_1', 'z': 'A_2'}
symbols = {sf: sp.Symbol(f'{nametable[sf]}', real=True) for sf in nametable.keys()}
display(flint_to_sympy(eq, nametable))
# print(*[s.is_real for s in symbols.values()])


{(1, 1, 1): 4/5, (0, 1, 2): 1, (0, 0, 3): 2}
3


4*A_0*A_1*A_2/5 + A_1*A_2**2 + 2*A_2**3

In [4]:
# forward conversion, sympy to flint
x, y, z = sp.symbols(['x', 'y', 'z'], real=True)
syms = [x, y, z]
ctx = flint.fmpq_mpoly_ctx.get(('x', 'y', 'z'), ordering='lex')
eq = x**2 + 3 * x * y

def sympy_to_flint(eq, ctx, syms):
    poly = sp.Poly(sp.expand(eq), *syms)
    n = ctx.nvars()
    dct = dict()
    for m, c in poly.terms():
        if not sp.nsimplify(c).is_Rational:
            raise ValueError(f'c={c} is not rational')
        dct[tuple(list(m) + [0]*(n - len(m)))] = flint.fmpq(int(c.p), int(c.q))
    return ctx.from_dict(dct)

flint_eq = sympy_to_flint(eq, ctx, syms)
print(flint_eq.context())

fmpq_mpoly_ctx(3, '<Ordering.lex: 'lex'>', ('x', 'y', 'z'))


In [8]:
# ---- Functional equation system generator from scratch ----

name = 'KP_real'
ans = family(name)
for (a, b), M in ans.hdict.items():
    print(f'h^{(a, b)}=')
    display(M)
syms = ans.syms
eqsys = EquationSystem(ans, htilde=None, q5_form = 'irf', order = 'degrevlex')
A = eqsys.alg
eqs = eqsys.generate()
eqs_red = eqsys.linear_reduce()
nametable = eqsys.nametable

x, y, z = syms[:3]
probe_eqs = [x**2 + 3 * x * y]
sol = Solver(ans, extra_equations=probe_eqs)
flint_eq = sol.sympy_to_flint(probe_eqs[0])
print(flint_eq)

# writing the equations in file
write = True
if write:
    filename =f'eqs_out_{name}.txt' 
    with open(f'tex/{filename}', 'w') as file:
        t0 = time.time()
        for eq in eqs_red:
            string = '\\begin{dmath}' + str(sp.latex(flint_to_sympy(eq, nametable))) + '=0' + '\\end{dmath}' + '\n'
            file.write(string)
        t1 = time.time()
        print(f'finished writing file {filename}, time required: {t1-t0}')


[NCq3Ansatz] {'name': 'KP_real', 'nsyms': 38, 'd': 3, 'lin_forms': 49}
h^(0, 0)=


Matrix([
[u_00^00, u_01^00, u_02^00],
[u_01^00, u_11^00, u_12^00],
[u_02^00, u_12^00, u_22^00]])

h^(0, 1)=


Matrix([
[u_00^01, u_01^01,       0],
[u_01^01, u_11^01,       0],
[      0,       0, u_22^01]])

h^(0, 2)=


Matrix([
[u_00^02,       0, u_02^02],
[      0, u_11^02,       0],
[u_02^02,       0, u_22^02]])

... (9 outputs left out)

Matrix([
[u_00^21,       0,       0],
[      0, u_11^21, u_12^21],
[      0, u_12^21, u_22^21]])

h^(2, 2)=


Matrix([
[u_00^22,       0, u_02^22],
[      0, u_11^22,       0],
[u_02^22,       0, u_22^22]])

[NCq3 EqSystem] eq generation: {'nvars': 38, 'n_eqs_raw': 351, 't_comms': 0.5365731716156006, 't_canonical': 0.04446125030517578, 'n_terms_res': 2691, 'max_degree': 3, 'n_monoms': 781}
[NCq3 EqSystem] linear reduce: {'shape': (351, 781), 'before': 351, 'after': 141, 't_rref': 0.06258511543273926}
x0^2 + 3*x0*x1
finished writing file eqs_out_KP_real.txt, time required: 0.3092920780181885


In [ ]:
# hdict = hdict_builder(family='dipolar_real', reflection=True, pattern=None, prefix='S', basis_name='SPIN1')
# ans = Ansatz(hdict, 3)
name = 'C'
ans = family(name)
eqsys = EquationSystem(ans, htilde=None, q5_form = 'irf', order = 'degrevlex')
A = eqsys.alg
for n, x in enumerate(zip(eqsys.var_names, eqsys.names)):
    if n > 5: break
    # print(x)

# print(f'polynomial context: {eqsys.ctx}')
# print(f'nvars: {eqsys.ctx.nvars()}')
eqs = eqsys.generate()
eqs_red = eqsys.linear_reduce()
# print(f'num of eqs: {len(eqs)} -> {len(eqs_red)}')
nametable = eqsys.nametable()

# writing the equations in file
filename =f'eqs_out_{name}.txt' 
with open(f'tex/{filename}', 'w') as file:
    t0 = time.time()
    for eq in eqs_red:
        string = '\\begin{dmath}' + str(sp.latex(flint_to_sympy(eq, nametable))) + '=0' + '\\end{dmath}' + '\n'
        file.write(string)
    t1 = time.time()
    print(f'finished writing file {filename}, time required: {t1-t0}')

e_syms = eqs_red[0].context().names()
# for n, e in enumerate(e_syms):
#     if n > 5: break
#     print(f'CAS var: {e_syms[n]}, symbolic var: {nametable[e_syms[n]]}')

[NCq3Ansatz] {'name': 'C', 'nsyms': 54, 'd': 3, 'lin_forms': 81}
[NCq3 EqSystem] eq generation: {'nvars': 54, 'n_eqs_raw': 672, 't_comms': 2.0825517177581787, 't_canonical': 0.35936522483825684, 'n_terms_res': 6561, 'max_degree': 3, 'n_monoms': 12310}
[NCq3 EqSystem] linear reduce: {'shape': (672, 12310), 'before': 672, 'after': 345, 't_rref': 19.059258222579956}
finished writing file eqs_out_C.txt, time required: 217.9849967956543


In [ ]:
# symmetries for families
G = symmetries_for('A')
print(*[g['mid'] for g in G], sep='\n')

None
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


From this point on, actual results will be computed

In [ ]:
with open(f'/Users/kristof/main/research/integrable_systems/NCq3_calculations/output_viewer/Z3_RR_solutions.txt', 'a') as file:
    file.write('\tmaki')